# 表の内容について質問に答える

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/19_qa/table_qa.ipynb)

NLP クックブック 19-07 ／ 質問応答

表 (スプレッドシートや DataFrame) の内容について、自然言語の質問に答えたい。「合計は？」「一番大きいのは？」のような集計も扱いたい。

In [ ]:
import pandas as pd
import torch
from transformers import TapasForQuestionAnswering, TapasTokenizer

name = "google/tapas-base-finetuned-wtq"    # 表に対する質問応答のモデル (英語)。セルの選択と集計の種類を予測する
tapas_tok = TapasTokenizer.from_pretrained(name)
tapas = TapasForQuestionAnswering.from_pretrained(name).eval()

table = pd.DataFrame({"City": ["Tokyo", "Osaka", "Nagoya", "Sapporo"], "Prefecture": ["Tokyo", "Osaka", "Aichi", "Hokkaido"],
                      "Population": ["13960000", "2750000", "2330000", "1970000"], "Area": ["2194", "225", "326", "1121"]})   # TAPAS にはすべて文字列で渡す
queries = ["Which city has the largest population?", "What is the total population of Osaka and Nagoya?",
           "How many cities have an area larger than 300?", "What is the average area?", "Which prefecture is Nagoya in?"]

enc = tapas_tok(table=table, queries=queries, padding="max_length", return_tensors="pt")
with torch.no_grad():
    out = tapas(**enc)
coords, aggs = tapas_tok.convert_logits_to_predictions(enc, out.logits, out.logits_aggregation)
AGG = {0: "NONE", 1: "SUM", 2: "AVERAGE", 3: "COUNT"}
for q, cells, agg in zip(queries, coords, aggs):
    values = [table.iat[c] for c in cells]
    nums = [float(v) for v in values if v.replace(".", "").isdigit()]
    result = {"NONE": ", ".join(values), "SUM": sum(nums), "AVERAGE": sum(nums) / max(1, len(nums)), "COUNT": len(values)}[AGG[agg]]
    print(f"{q}\n   {AGG[agg]} {values} → {result}")

In [ ]:
# 日本語の表: 表を Markdown にして LLM に読ませる
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-3b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
llm = AutoModelForCausalLM.from_pretrained(name).eval()

df = pd.DataFrame({"都市": ["東京", "大阪", "名古屋", "札幌"], "都道府県": ["東京都", "大阪府", "愛知県", "北海道"],
                   "人口 (万人)": [1396, 275, 233, 197], "面積 (km2)": [2194, 225, 326, 1121]})
markdown = "| " + " | ".join(df.columns) + " |\n|" + "---|" * len(df.columns) + "\n" + "\n".join("| " + " | ".join(map(str, r)) + " |" for r in df.values)

def ask(question):
    prompt = f"次の表だけを見て、質問に短く答えてください。\n\n{markdown}\n\n質問: {question}"
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": prompt}], add_generation_prompt=True, return_tensors="pt", return_dict=True)
    with torch.no_grad():
        out = llm.generate(**inputs, max_new_tokens=60, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

checks = {"人口が一番多い都市はどこですか？": df.loc[df["人口 (万人)"].idxmax(), "都市"], "名古屋は何県にありますか？": "愛知県",
          "面積が 300 km2 より大きい都市はいくつありますか？": str((df["面積 (km2)"] > 300).sum()),
          "大阪と名古屋の人口の合計は何万人ですか？": str(275 + 233), "4 都市の面積の平均は？": str(df["面積 (km2)"].mean())}
for q, truth in checks.items():
    print(f"{q}\n   LLM: {ask(q)}\n   pandas で計算した正解: {truth}")

- TAPAS は、答えに使うセルと集計の種類 (なし・合計・平均・個数) を予測する。計算そのものはプログラムが行うので、選んだセルが正しければ数値は必ず合う。
- LLM に表を読ませる方法は手軽で日本語でも使えるが、セルの参照は得意でも、合計や平均などの計算は間違えやすい。計算が要る質問は、上のように pandas の結果と突き合わせて確かめる。
- 行数の多い表はプロンプトに入りきらない。質問から pandas のコードや SQL を生成させて実行する方法 (text-to-SQL) が、大きな表では現実的。
- 表を文字列にする形式 (Markdown、CSV、HTML) で精度が変わる。列名と単位を明記するのが基本。